# Day 063 — Exercise 3: process_webhook_event

Stripe notifies your server about payment events via **webhooks** — HTTP POST requests sent to your `/webhook` endpoint whenever something happens (subscription created, payment failed, etc.). Your server updates its database based on the event type.

Key events for a subscription product:
| Event | Meaning | Action |
|-------|---------|--------|
| `customer.subscription.created` | User subscribed | Set plan = pro |
| `customer.subscription.deleted` | User cancelled | Set plan = free |
| `invoice.payment_failed` | Payment declined | Mark as past_due |

## Task

Implement `process_webhook_event(event_type, payload, user_db) -> dict`:

- `payload['customer_id']` identifies the user
- Missing customer_id → `{success: False, action: 'no_customer_id'}`
- Unknown customer → `{success: False, action: 'customer_not_found'}`
- Event in `PLAN_EVENTS` → update `user_db[cid]['plan']`
- `invoice.payment_failed` → set `user_db[cid]['status'] = 'past_due'`
- Any other event → `{success: True, action: 'ignored'}`

## Your Implementation

In [ ]:
PLAN_EVENTS = {
    "customer.subscription.created": "pro",
    "customer.subscription.updated": "pro",
    "customer.subscription.deleted": "free",
}

def process_webhook_event(event_type: str, payload: dict,
                          user_db: dict) -> dict:
    """Handle a Stripe webhook event and update user_db.

    payload keys:
        customer_id  str — identifies the user in user_db

    Behaviour:
    - Missing customer_id in payload
        → {success: False, action: 'no_customer_id', customer_id: ''}
    - customer_id not in user_db
        → {success: False, action: 'customer_not_found', customer_id: ...}
    - event_type in PLAN_EVENTS
        → set user_db[customer_id]['plan'] = PLAN_EVENTS[event_type]
        → {success: True, action: f'plan_set_{new_plan}', customer_id: ...}
    - event_type == 'invoice.payment_failed'
        → set user_db[customer_id]['status'] = 'past_due'
        → {success: True, action: 'status_set_past_due', customer_id: ...}
    - any other event_type
        → {success: True, action: 'ignored', customer_id: ...}
    """
    # TODO: implement the event routing logic
    raise NotImplementedError


In [ ]:
PLAN_EVENTS = {
    "customer.subscription.created": "pro",
    "customer.subscription.updated": "pro",
    "customer.subscription.deleted": "free",
}

def process_webhook_event(event_type, payload, user_db):
    customer_id = payload.get("customer_id", "")
    if not customer_id:
        return {"success": False, "action": "no_customer_id", "customer_id": ""}
    if customer_id not in user_db:
        return {"success": False, "action": "customer_not_found",
                "customer_id": customer_id}
    if event_type in PLAN_EVENTS:
        new_plan = PLAN_EVENTS[event_type]
        user_db[customer_id]["plan"] = new_plan
        return {"success": True, "action": f"plan_set_{new_plan}",
                "customer_id": customer_id}
    if event_type == "invoice.payment_failed":
        user_db[customer_id]["status"] = "past_due"
        return {"success": True, "action": "status_set_past_due",
                "customer_id": customer_id}
    return {"success": True, "action": "ignored", "customer_id": customer_id}


## Automated checks

In [ ]:
score, total = 0, 6
try:
    db = {
        "cus_123": {"plan": "free", "status": "active"},
        "cus_456": {"plan": "pro",  "status": "active"},
    }

    # subscription.created → pro
    r = process_webhook_event("customer.subscription.created",
                              {"customer_id": "cus_123"}, db)
    assert r["success"] is True and r["action"] == "plan_set_pro"
    assert db["cus_123"]["plan"] == "pro"
    score += 1; print("\u2705 subscription.created upgrades plan to pro")

    # subscription.deleted → free
    r2 = process_webhook_event("customer.subscription.deleted",
                               {"customer_id": "cus_456"}, db)
    assert r2["success"] is True and r2["action"] == "plan_set_free"
    assert db["cus_456"]["plan"] == "free"
    score += 1; print("\u2705 subscription.deleted downgrades plan to free")

    # invoice.payment_failed → status = past_due
    r3 = process_webhook_event("invoice.payment_failed",
                               {"customer_id": "cus_123"}, db)
    assert r3["success"] is True and r3["action"] == "status_set_past_due"
    assert db["cus_123"]["status"] == "past_due"
    score += 1; print("\u2705 invoice.payment_failed marks status as past_due")

    # unknown customer
    r4 = process_webhook_event("customer.subscription.created",
                               {"customer_id": "cus_999"}, db)
    assert r4["success"] is False and r4["action"] == "customer_not_found"
    score += 1; print("\u2705 unknown customer_id → success=False")

    # missing customer_id
    r5 = process_webhook_event("customer.subscription.created", {}, db)
    assert r5["success"] is False and r5["action"] == "no_customer_id"
    score += 1; print("\u2705 missing customer_id → success=False")

    # unknown event → ignored
    r6 = process_webhook_event("payment_intent.created",
                               {"customer_id": "cus_123"}, db)
    assert r6["success"] is True and r6["action"] == "ignored"
    score += 1; print("\u2705 unknown event type → ignored (success=True)")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
PLAN_EVENTS = {
    "customer.subscription.created": "pro",
    "customer.subscription.updated": "pro",
    "customer.subscription.deleted": "free",
}

def process_webhook_event(event_type, payload, user_db):
    customer_id = payload.get("customer_id", "")
    if not customer_id:
        return {"success": False, "action": "no_customer_id", "customer_id": ""}
    if customer_id not in user_db:
        return {"success": False, "action": "customer_not_found",
                "customer_id": customer_id}
    if event_type in PLAN_EVENTS:
        new_plan = PLAN_EVENTS[event_type]
        user_db[customer_id]["plan"] = new_plan
        return {"success": True, "action": f"plan_set_{new_plan}",
                "customer_id": customer_id}
    if event_type == "invoice.payment_failed":
        user_db[customer_id]["status"] = "past_due"
        return {"success": True, "action": "status_set_past_due",
                "customer_id": customer_id}
    return {"success": True, "action": "ignored", "customer_id": customer_id}
```

**Why `ignored` not an error for unknown events?** Stripe sends many event types (30+). Your server should handle the ones it cares about and silently accept the rest. Returning an error for unknown events would cause Stripe to retry the webhook repeatedly, flooding your logs.

</details>